# 05 — Large-data performance and windowed access

**Goal:** never load a full frame when you need a window. A 20 m GCOV frame is ~18 000 x 18 000
pixels per layer (≈ 1.2 GB per layer as float32, several GB for all layers): it does not fit
comfortably in the memory of a training laptop, and it is never needed.

HDF5 stores each layer in **chunks** (512 x 512 here). A window read decompresses only the chunks
it overlaps; a strided read (`a[::10, ::10]`) still touches every chunk and is slow.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nisar_s1.config import load_kit
from nisar_s1 import product as P

kit = load_kit()                                  # training/notebooks/kit_config.yaml
OUT = Path(kit["data_dir"]) / "outputs" / "05"   # outputs stay outside the repository
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
session = P.load_session(kit["session_file"], required=("granule", "frequency"))
src_ctx = lambda: P.open_local_or_remote(session["gcov_local"], session["gcov_url"])
import time

In [2]:
with src_ctx() as h5:
    prof = P.product_profile(h5)
    grid = prof.grids[session["frequency"]]
    d = h5[f"{grid.path}/HVHV"]
    full_gb = np.prod(d.shape) * d.dtype.itemsize / 2**30
    print(f"HVHV full layer: {d.shape}, {full_gb:.2f} GB in memory; chunks {d.chunks}")
    print("memory guard refuses it:", end=" ")
    try:
        P.read_layer(h5, grid, "HVHV", P.window_from_bounds(grid, grid.bounds), max_gb=1.0)
    except MemoryError as e:
        print(e)

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5
HVHV full layer: (17928, 18108), 1.21 GB in memory; chunks (512, 512)
memory guard refuses it: window needs 1.21 GB > max_gb=1.0; shrink the AOI or raise max_gb


## Aligned versus misaligned windows

In [3]:
def timed(h5, rows, cols):
    t = time.perf_counter()
    a = h5[f"{grid.path}/HVHV"][rows, cols]
    return time.perf_counter() - t, a.nbytes / 2**20

res = []
with src_ctx() as h5:
    r0, c0 = (grid.shape[0] // 2 // 512) * 512, (grid.shape[1] // 2 // 512) * 512
    for name, rows, cols in [
        ("1024² aligned on chunks (4 chunks)", slice(r0, r0 + 1024), slice(c0, c0 + 1024)),
        ("1024² shifted by 256 px (9 chunks)", slice(r0 + 256, r0 + 1280), slice(c0 + 256, c0 + 1280)),
        ("strided 1/16 over 16384² (~1024² values)", slice(r0 - 8192, r0 + 8192, 16), slice(c0 - 8192, c0 + 8192, 16)),
    ]:
        rows = slice(max(rows.start, 0), min(rows.stop, grid.shape[0]), rows.step)
        cols = slice(max(cols.start, 0), min(cols.stop, grid.shape[1]), cols.step)
        s, mb = timed(h5, rows, cols)
        res.append({"read": name, "seconds": round(s, 2), "MB returned": round(mb, 1)})
pd.DataFrame(res)

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5


,read,seconds,MB returned
0,1024² aligned on chunks (4 chunks),0.15,4.0
1,1024² shifted by 256 px (9 chunks),0.16,4.0
2,strided 1/16 over 16384² (~1024² values),9.50,4.0


The strided read returns as many values as the 1024² window but has to decompress hundreds of chunks.
For an **overview** of a whole frame, use the browse products (PNG/KML shipped next to the HDF5),
or the coarser frequency B grid, rather than striding through frequency A.

## Tile-wise processing
When an AOI is too large for memory, process it tile by tile (`P.block_iter`) and keep only
summary values. Here: the mean HV of a 4096² area in 1024² tiles, never holding more than one tile.

In [4]:
big = P.Window(slice(r0 - 2048, r0 + 2048), slice(c0 - 2048, c0 + 2048), grid.transform, f"EPSG:{grid.epsg}")
tiles = list(P.block_iter(big, 1024))
s, n = 0.0, 0
t = time.perf_counter()
with src_ctx() as h5:
    for i, w in enumerate(tiles, 1):
        a = P.read_layer(h5, grid, "HVHV", w)
        s += np.nansum(a); n += np.isfinite(a).sum()
        if i % 4 == 0: print(f"[{i}/{len(tiles)}] tiles done")
print(f"mean HV over {n:,} pixels: {10*np.log10(s/n):.2f} dB in {time.perf_counter()-t:.1f} s")

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5
[4/16] tiles done


[8/16] tiles done
[12/16] tiles done
[16/16] tiles done
mean HV over 16,777,179 pixels: -11.55 dB in 0.4 s


## Remote access
The same code reads a file **on the ASF server** through HTTPS range requests (only the needed
chunks travel over the network). Set `TRY_REMOTE = True` to time a remote 512² read (needs
Earthdata credentials in `~/.netrc` and internet).

In [5]:
TRY_REMOTE = False
if TRY_REMOTE:
    from nisar_s1.io_nisar import open_product
    t = time.perf_counter()
    with open_product(session["gcov_url"]) as h5:
        a = h5[f"{grid.path}/HVHV"][r0:r0 + 512, c0:c0 + 512]
    print(f"remote 512² read: {time.perf_counter() - t:.1f} s")

In [6]:
checks = {
    "full-layer read refused by the guard": full_gb > 1.0,
    "aligned window faster than strided read": res[0]['seconds'] < res[2]['seconds'],
    "tile-wise mean computed": n > 0,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
assert all(checks.values()), "some checks failed: read the messages above"
print("Module 05 STATUS: PASS")

  [x] full-layer read refused by the guard
  [x] aligned window faster than strided read
  [x] tile-wise mean computed
Module 05 STATUS: PASS
